In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [5]:
df=pd.read_csv('/content/SMSSpamCollection', sep='\t')

In [6]:
df.head()

,spam_desc,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df['spam_desc'] = df['spam_desc'].map({
    'ham': 0,
    'spam': 1
})

In [8]:
from sklearn.model_selection import train_test_split

x = df['text']
y = df['spam_desc']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [9]:
x.head()

,text
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [11]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

models=[MultinomialNB(),LogisticRegression()]


# model = MultinomialNB()
# model.fit(x_train_tfidf, y_train)

In [12]:

for model in models:
  print(model," ********************* ")
  model.fit(x_train_tfidf, y_train)
  pred = model.predict(x_test_tfidf)

  from sklearn.metrics import accuracy_score, classification_report

  print(accuracy_score(y_test, pred))
  print(classification_report(y_test, pred))

MultinomialNB()  ********************* 
0.9742822966507177
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1448
           1       1.00      0.81      0.89       224

    accuracy                           0.97      1672
   macro avg       0.99      0.90      0.94      1672
weighted avg       0.98      0.97      0.97      1672

LogisticRegression()  ********************* 
0.965311004784689
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      1448
           1       1.00      0.74      0.85       224

    accuracy                           0.97      1672
   macro avg       0.98      0.87      0.92      1672
weighted avg       0.97      0.97      0.96      1672



In [13]:
!pip install gensim
from gensim.models import Word2Vec

In [14]:
df['text'] = df['text'].str.lower()

sentences = df['text'].apply(lambda x: x.split())

In [18]:
type(sentences)

pandas.core.series.Series

In [19]:
sentences[0]

['go',
 'until',
 'jurong',
 'point,',
 'crazy..',
 'available',
 'only',
 'in',
 'bugis',
 'n',
 'great',
 'world',
 'la',
 'e',
 'buffet...',
 'cine',
 'there',
 'got',
 'amore',
 'wat...']

In [15]:
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [16]:
def sentence_vector(sentence):
    words = [
        word for word in sentence
        if word in w2v_model.wv
    ]

    if len(words) == 0:
        return np.zeros(100)

    return np.mean(
        w2v_model.wv[words],
        axis=0
    )

In [21]:
X = np.array(
    [sentence_vector(sentence)
     for sentence in sentences]
)

y = df['spam_desc']

In [23]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=12,
    stratify=y
)

In [24]:
from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
# from sklearn.naive_bayes import GaussianNB
# from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = [
    LogisticRegression(),
    # LinearSVC(),
    RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    # GaussianNB(),
    # MLPClassifier(
    #     hidden_layer_sizes=(128, 64),
    #     max_iter=500,
    #     random_state=42
    # ),
    XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5
    ),
    LGBMClassifier(
        n_estimators=300
    ),
]

In [25]:

for model in models:
  print(model," ********************* ")
  model.fit(x_train_tfidf, y_train)
  pred = model.predict(x_test_tfidf)

  from sklearn.metrics import accuracy_score, classification_report

  print(accuracy_score(y_test, pred))
  print(classification_report(y_test, pred))

LogisticRegression()  ********************* 
0.8660287081339713
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      1448
           1       0.00      0.00      0.00       224

    accuracy                           0.87      1672
   macro avg       0.43      0.50      0.46      1672
weighted avg       0.75      0.87      0.80      1672

RandomForestClassifier(n_estimators=200, random_state=42)  ********************* 


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.8415071770334929
              precision    recall  f1-score   support

           0       0.87      0.97      0.91      1448
           1       0.15      0.04      0.06       224

    accuracy                           0.84      1672
   macro avg       0.51      0.50      0.49      1672
weighted avg       0.77      0.84      0.80      1672

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraint

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [26]:
# tfidf outperforms all in terms of accuracy